In [1]:
import pandas as pd
import ast
import numpy as np

questions = pd.read_csv('../Dataset/question_bank.csv')

# Rename 'Question ID' to 'question_id' if it exists to match our expected format
if 'Question ID' in questions.columns:
    questions = questions.rename(columns={'Question ID': 'question_id'})

# Ensure no duplication of rows and embeddings computed once per question_id
questions = questions.drop_duplicates(subset=['question_id']).reset_index(drop=True)

# Clean options into a Python list format safely
def clean_options(opt):
    if isinstance(opt, str):
        try:
            return ast.literal_eval(opt)
        except (ValueError, SyntaxError):
            return [o.strip() for o in opt.split('|')]
    return opt if isinstance(opt, list) else []

questions['options_cleaned'] = questions['options'].apply(clean_options)

# Dynamically add num_options and avg_option_length
questions['num_options'] = questions['options_cleaned'].apply(len)
questions['avg_option_length'] = questions['options_cleaned'].apply(
    lambda x: np.mean([len(str(o)) for o in x]) if len(x) > 0 else 0
)

# Keep the original specified columns alongside the newly engineered ones (deleting original 'options')
expected_cols = [
    'question_id', 'question_text', 'correct_answer', 'difficulty', 'num_words', 'qstn_complexity'
]
# Avoid KeyError in case the dataset perfectly matching those names
existing_cols = [c for c in expected_cols if c in questions.columns]
questions = questions[existing_cols + ['options_cleaned', 'num_options', 'avg_option_length']]

questions.head()

,question_id,question_text,correct_answer,difficulty,num_words,qstn_complexity,options_cleaned,num_options,avg_option_length
0,1,Which of the following activities is a data mi...,Predicting the future stock price of a company...,hard,10,8.370000,[A. Dividing the customers of a company accord...,4,60.25
1,2,What is the range of values possible for the c...,"[-1,1]",easy,11,5.863636,"[A. [0,1], B. [-1,1], C. [0, infinity [, D. [-...",4,13.00
2,3,Is the cosine measure sensitive to scaling ?,No,easy,8,7.368571,"[A. Yes, B. No]",2,5.50
3,4,Is the Euclidean distance sensitive to scaling ?,Yes,easy,8,7.368571,"[A. Yes, B. No]",2,5.50
4,5,"If two objects have a cosine measure of 1, are...",No,medium,12,6.790000,"[A. Yes, B. No]",2,5.50


In [2]:
# Ensure sentence-transformers is installed if running into a ModuleNotFoundError
# %pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# Initialize the model 
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute text embeddings for question_text
# This prevents data leakage by purely encoding the text into an initial state array
embeddings = model.encode(questions['question_text'].tolist(), show_progress_bar=True)

# Store embeddings as a separate column where each element is explicitly a numpy array
# Note: In pandas, a column containing numpy arrays will always report its dtype as 'object',
# because it is storing pointers to the array objects. The elements themselves are np.ndarrays.
questions['embedding'] = list(embeddings)

print("Embeddings shape:", embeddings.shape)
print("Type of a single element in the embedding column:", type(questions['embedding'].iloc[0]))
questions[['question_text', 'embedding']].head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings shape: (150, 384)
Type of a single element in the embedding column: <class 'numpy.ndarray'>


,question_text,embedding
0,Which of the following activities is a data mi...,"[0.020898055, 0.016404873, -0.07365863, -0.039..."
1,What is the range of values possible for the c...,"[-0.008224491, 0.05351976, -0.10834609, -0.035..."
2,Is the cosine measure sensitive to scaling ?,"[0.00035251075, 0.009688629, -0.06381428, 0.00..."
3,Is the Euclidean distance sensitive to scaling ?,"[0.061683588, -0.03877466, -0.0722496, -0.0858..."
4,"If two objects have a cosine measure of 1, are...","[-0.03584731, -0.003314979, -0.05435998, -0.00..."


In [5]:
questions.info()

questions.to_csv('../Dataset/questions.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   question_id        150 non-null    int64  
 1   question_text      150 non-null    object 
 2   correct_answer     150 non-null    object 
 3   difficulty         150 non-null    object 
 4   num_words          150 non-null    int64  
 5   qstn_complexity    150 non-null    float64
 6   options_cleaned    150 non-null    object 
 7   num_options        150 non-null    int64  
 8   avg_option_length  150 non-null    float64
 9   embedding          150 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 11.8+ KB
